# 🎙️ Voice Over Studio (Bahasa Indonesia) — v2 (tempo diperbaiki)

Perbaikan dari versi sebelumnya: diproses **per kalimat** agar tidak
menumpuk, jeda antar kalimat bisa diatur, `remove_silence` dimatikan,
dan teks dibersihkan otomatis (em dash → koma).

**Cara pakai:**
1. **Runtime → Change runtime type → T4 GPU → Save**.
2. **Runtime → Run all**.
3. Klik link **`https://xxxx.gradio.live`** di bawah sel ke-2 untuk membuka aplikasi.

> Untuk hasil terbaik: isi **Transkrip sampel** (kalimat persis di rekaman),
> pakai sampel bertempo tenang 10-15 detik.

> Gunakan hanya suara Anda sendiri / narator yang sudah mengizinkan.

In [ ]:
# 1) Pasang program (tunggu 2-4 menit)
import subprocess, sys
print('Memasang F5-TTS + Gradio, mohon tunggu...')
subprocess.run([sys.executable,'-m','pip','install','-q','f5-tts','gradio'])
subprocess.run(['bash','-c','apt-get -qq install -y ffmpeg >/dev/null 2>&1'])
print('OK, pemasangan selesai.')

In [ ]:
# 2) Jalankan aplikasi (muat model + tampilkan antarmuka)
import re, numpy as np, torch, gradio as gr
from huggingface_hub import hf_hub_download
from f5_tts.api import F5TTS

print("Mengunduh & memuat model Indonesia (1-2 menit, sekali saja)...")
ckpt = hf_hub_download('Eempostor/F5-TTS-INDO-FINETUNE-V2', 'f5_tts_indo_v2.pt')
vocab = hf_hub_download('Eempostor/F5-TTS-INDO-FINETUNE-V2', 'vocab.txt')
MODEL = F5TTS(model='F5TTS_v1_Base', ckpt_file=ckpt, vocab_file=vocab)
print("Model siap.")

try:
    from f5_tts.infer.utils_infer import preprocess_ref_audio_text
except Exception:
    preprocess_ref_audio_text = None

def bersihkan(teks):
    # em dash / en dash -> koma; buang tanda kutip nyasar; rapikan spasi
    teks = teks.replace('—', ', ').replace('–', ', ')
    for q in ['“', '”', '‘', '’', '"', "'"]:
        teks = teks.replace(q, '')
    teks = re.sub(r'[ \t]+', ' ', teks)
    return teks.strip()

def pecah_kalimat(teks):
    # pecah per baris lalu per akhiran kalimat (. ! ?)
    kalimat = []
    for baris in teks.split('\n'):
        baris = baris.strip()
        if not baris:
            continue
        for k in re.split(r'(?<=[\.\!\?])\s+', baris):
            k = k.strip()
            if k:
                kalimat.append(k)
    return kalimat

def buat_suara(ref_audio, ref_text, naskah, kecepatan, jeda, nfe, progress=gr.Progress()):
    if ref_audio is None:
        raise gr.Error("Rekam atau unggah sampel suara dulu.")
    naskah = bersihkan(naskah or '')
    if not naskah:
        raise gr.Error("Tulis naskah voice over dulu.")

    rt = (ref_text or '').strip()
    ref = ref_audio
    # siapkan referensi + transkrip sekali saja (ASR otomatis kalau transkrip kosong)
    if preprocess_ref_audio_text is not None:
        try:
            ref, rt = preprocess_ref_audio_text(ref_audio, rt)
        except Exception:
            pass

    kalimat = pecah_kalimat(naskah)
    sr = 24000
    jeda_samples = np.zeros(int(24000 * float(jeda)), dtype=np.float32)
    potongan = []
    for i, k in enumerate(kalimat):
        progress((i + 1) / len(kalimat), desc=f"Kalimat {i+1}/{len(kalimat)}")
        wav, sr, _ = MODEL.infer(
            ref_file=ref, ref_text=rt, gen_text=k,
            speed=float(kecepatan), nfe_step=int(nfe),
            remove_silence=False, cross_fade_duration=0.0,
        )
        potongan.append(np.asarray(wav, dtype=np.float32))
        potongan.append(jeda_samples)
    full = np.concatenate(potongan) if potongan else np.zeros(1, dtype=np.float32)
    return (sr, full)

with gr.Blocks(title="Voice Over Studio v2 - Portal BMP") as demo:
    gr.Markdown("# 🎙️ Voice Over Studio (Bahasa Indonesia) — v2\n"
                "Diproses per kalimat agar tempo rapi. **Gunakan hanya suara Anda "
                "sendiri atau narator yang sudah mengizinkan.**")
    with gr.Row():
        with gr.Column():
            ref = gr.Audio(label="1. Sampel suara (rekam / unggah, 10-15 detik, tempo tenang)",
                           sources=["microphone", "upload"], type="filepath")
            reft = gr.Textbox(label="Transkrip sampel (SANGAT disarankan diisi persis)",
                              placeholder="Kalimat persis yang diucapkan di sampel...")
            naskah = gr.Textbox(label="2. Naskah voice over", lines=8,
                                placeholder="Tempel naskah berita di sini...")
            with gr.Row():
                spd = gr.Slider(0.6, 1.2, value=0.9, step=0.05, label="Kecepatan")
                jeda = gr.Slider(0.0, 0.8, value=0.25, step=0.05, label="Jeda antar kalimat (dtk)")
            nfe = gr.Slider(16, 64, value=32, step=4, label="Kualitas (NFE, makin tinggi makin halus tapi lambat)")
            btn = gr.Button("Buat Voice Over", variant="primary")
        with gr.Column():
            out = gr.Audio(label="Hasil voice over (putar & unduh)", type="numpy")
    btn.click(buat_suara, [ref, reft, naskah, spd, jeda, nfe], out)

demo.launch(share=True)
